# Tuning C30: Cấu hình chính đã chọn ở FE

Mục tiêu: tìm tham số riêng cho từng mô hình trên cấu hình C30 cố định.

Kết luận và tham số được chọn lấy từ bảng cuối notebook sau khi chạy. C30 là đường chính; A12/B27 kiểm tra đóng góp của nhóm đặc trưng sau tuning.

Cả ba gọi `src/tuning_cv.py`, dùng cohort và ba fold của FE, cùng danh sách ứng viên và ngân sách. Notebook không chấm validation/test. Các hàm nằm trong module dùng chung; mỗi ô code có giải thích ngay phía trước.

C30 đều dùng 22.503 giờ đủ history, cùng ba fold, chấm đúng 16.875 giờ OOF. Module gọi trực tiếp load_protocol và prepare_fold; không điền median cho lag.

## 1. Nạp module

Tìm gốc dự án từ CSV theo giờ. Mỗi notebook chạy bằng namespace mới để không mang trạng thái của lượt tuning khác.

In [5]:
from pathlib import Path
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'data/preprocessed/hourly_unfilled.csv').is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import pandas as pd
from IPython.display import Markdown, display
from src import tuning_cv as cv
CONFIG = 'C30'
CONFIGS = (CONFIG,)
features = cv.CONFIGS[CONFIG]


## 2. Dùng cohort và fold của FE

`load_context` gọi `load_protocol` rồi `prepare_fold` trong `src/feature_experiments.py`. Mọi cấu hình đều lọc cùng các giờ có đủ lag_24h, lag_168h và mean_24h, kể cả A12/B27. Không điền median cho lag. `prepare_fold` chỉ học median thời tiết từ đúng fold-train, giữ các quan sát weather gốc trước t làm ngữ cảnh.

Ma trận chung có 30 cột không thiếu; mỗi mô hình chỉ chọn đúng cột của CONFIG. Ridge có thêm `weekday_hour = day_of_week*24 + hour`, rồi one-hot thứ–giờ và tháng trong fold. Đây là đổi biểu diễn lịch, không thêm thông tin ngoài lịch. Các cột số của Ridge và đầu vào MLP được chuẩn hóa chỉ trên fold-train.

In [6]:
context = cv.load_context(project_root)
display(pd.DataFrame([dict(config=CONFIG, n_features=len(features), n=len(context['cohort']),
                           n_train=len(context['train']), n_oof=int(context['manifest'].n.sum()),
                           protocol_hash=context['protocol_hash'])]))
display(cv.describe_folds(context))
display(cv.budget_table(context))
display(Markdown('**Sáu ứng viên RF được lấy trực tiếp từ lưới:**'))
display(pd.DataFrame(context['tuning_protocol']['candidates']['Random Forest']))

,config,n_features,n,n_train,n_oof,protocol_hash
0,C30,30,22503,28402,16875,6042efc8e56db7968fddaa33ee9d8b43d53c817fb778b1...


,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...


,model,candidates,folds,fits_per_config,n
0,Ridge,6,3,18,22503
1,Random Forest,6,3,18,22503
2,Gradient Boosting,6,3,18,22503
3,MLP,6,3,18,22503
4,XGBoost,6,3,18,22503


**Sáu ứng viên RF được lấy trực tiếp từ lưới:**

,model__n_estimators,model__min_samples_leaf,model__max_features,model__max_depth
0,100,1,0.80,8
1,100,5,0.33,8
2,100,10,1.00,12
3,200,1,0.50,16
4,400,5,0.33,16
5,200,2,0.80,20


## 3. Chạy CV và hai baseline

Mỗi mô hình dùng sáu ứng viên × ba fold, cùng danh sách giữa A/B/C. Mỗi fold tạo pipeline mới. Điền thiếu đã do FE xử lý nên pipeline không có imputer thứ hai. Chọn MAE trung bình fold, RMSE phá hòa; đồng thời lưu R², SD và cảnh báo hội tụ.

Hai baseline cũng chấm trên đúng các giờ này: trung bình thứ–giờ học từ fold-train và seasonal naive lấy lag_168h. OOF là dự đoán trên các phần score nằm ngoài phần học của từng fold.

Ô sau thực sự huấn luyện lại. Kết quả của riêng CONFIG ghi vào `outputs/CONFIG/results.json`, kèm `protocol.json`; chạy cấu hình này không ghi đè kết quả cấu hình khác. History ở score được cập nhật bằng các quan sát quá khứ đã đến; chưa đại diện dự báo cả đoạn tương lai từ một mốc.

In [7]:
result = cv.run_configuration(context, CONFIG)

C30 | Ridge | candidate 1/6 | fold 1: MAE=260.596
C30 | Ridge | candidate 1/6 | fold 2: MAE=275.841
C30 | Ridge | candidate 1/6 | fold 3: MAE=250.902
C30 | Ridge | candidate 2/6 | fold 1: MAE=260.534
C30 | Ridge | candidate 2/6 | fold 2: MAE=275.732
C30 | Ridge | candidate 2/6 | fold 3: MAE=250.854
C30 | Ridge | candidate 3/6 | fold 1: MAE=260.724
C30 | Ridge | candidate 3/6 | fold 2: MAE=274.782
C30 | Ridge | candidate 3/6 | fold 3: MAE=250.386
C30 | Ridge | candidate 4/6 | fold 1: MAE=272.036
C30 | Ridge | candidate 4/6 | fold 2: MAE=271.988
C30 | Ridge | candidate 4/6 | fold 3: MAE=248.532
C30 | Ridge | candidate 5/6 | fold 1: MAE=310.620
C30 | Ridge | candidate 5/6 | fold 2: MAE=289.135
C30 | Ridge | candidate 5/6 | fold 3: MAE=260.335
C30 | Ridge | candidate 6/6 | fold 1: MAE=359.799
C30 | Ridge | candidate 6/6 | fold 2: MAE=329.666
C30 | Ridge | candidate 6/6 | fold 3: MAE=298.387
C30 | Random Forest | candidate 1/6 | fold 1: MAE=245.579
C30 | Random Forest | candidate 1/6 | fold

## 4. Kết quả và tham số được chọn

MAE/RMSE có đơn vị xe/giờ, R² không có đơn vị. SD chỉ mô tả biến động giữa ba giai đoạn, không phải ngưỡng kiểm định thống kê. Các fold bằng số hàng chấm nhưng khác thời lượng lịch. Xem cảnh báo hội tụ trong JSON trước khi nhận xét MLP.

Kết quả chỉ dùng train CV; không tự fit lại hoặc đánh giá validation/test. Ensemble về sau đọc tham số và OOF đã lưu, không tìm lại tham số mô hình thành phần.

In [8]:
display(cv.results_table(result))
for model in result['results']:
    display(Markdown(f"**{model['model']}** — tham số được chọn: `{model['best_params']}`"))
    display(pd.DataFrame(model['folds']))
best = cv.results_table(result).loc[lambda t: ~t.model.isin(cv.BASELINES)].sort_values(['MAE_mean', 'RMSE_mean']).iloc[0]
display(Markdown(f"Trong {CONFIG}, mô hình có MAE CV thấp nhất là **{best['model']}**, MAE {best.MAE_mean:.3f}, RMSE {best.RMSE_mean:.3f} xe/giờ. Chưa phải kết quả holdout."))
rf = next(m for m in result['results'] if m['model'] == 'Random Forest')
display(Markdown('**Kiểm tra sáu ứng viên RF đã thực sự chạy:**'))
display(pd.DataFrame([dict(candidate=t['candidate'], **t['parameters'], MAE_CV=t['MAE_mean'], RMSE_CV=t['RMSE_mean'], n=sum(f['n'] for f in t['folds'])) for t in rf['candidates']]))

,config,model,n,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,C30,Mean weekday-hour,16875,305.628321,9.218800,512.521230,51.706045,0.931526,0.012548
1,C30,Seasonal naive 168h,16875,336.788859,25.656439,619.320594,79.984387,0.899629,0.023644
2,C30,Ridge,16875,261.964126,12.244862,417.423382,45.509656,0.954493,0.009382
3,C30,Random Forest,16875,231.039483,11.824613,381.359717,40.660592,0.962000,0.008075
4,C30,Gradient Boosting,16875,255.930858,10.168016,410.306333,40.627486,0.956038,0.008945
5,C30,MLP,16875,277.113358,19.057196,418.176048,38.930936,0.954423,0.007995
6,C30,XGBoost,16875,235.983530,11.368029,386.060040,36.681091,0.961106,0.007526


**Ridge** — tham số được chọn: `{'model__alpha': 0.1}`

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash,MAE,RMSE,R2,seconds,warnings
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...,260.724289,434.655832,0.952015,0.055724,[]
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...,274.781738,451.800306,0.946599,0.114779,[]
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...,250.386349,365.814009,0.964866,0.114533,[]


**Random Forest** — tham số được chọn: `{'model__n_estimators': 400, 'model__min_samples_leaf': 5, 'model__max_features': 0.33, 'model__max_depth': 16}`

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash,MAE,RMSE,R2,seconds,warnings
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...,226.441455,382.591604,0.962822,1.047828,[]
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...,244.472458,421.390367,0.953546,1.969667,[]
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...,222.204537,340.097180,0.969632,3.116201,[]


**Gradient Boosting** — tham số được chọn: `{'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.1}`

... (4 outputs left out)

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash,MAE,RMSE,R2,seconds,warnings
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...,229.736416,381.967836,0.962943,0.916778,[]
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...,249.105200,424.615631,0.952832,1.255187,[]
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...,229.108975,351.596652,0.967544,1.437470,[]


Trong C30, mô hình có MAE CV thấp nhất là **Random Forest**, MAE 231.039, RMSE 381.360 xe/giờ. Chưa phải kết quả holdout.

**Kiểm tra sáu ứng viên RF đã thực sự chạy:**

,candidate,model__n_estimators,model__min_samples_leaf,model__max_features,model__max_depth,MAE_CV,RMSE_CV,n
0,1,100,1,0.80,8,249.598760,404.668535,16875
1,2,100,5,0.33,8,248.429978,400.744464,16875
2,3,100,10,1.00,12,254.908535,429.116122,16875
3,4,200,1,0.50,16,232.233408,381.098614,16875
4,5,400,5,0.33,16,231.039483,381.359717,16875
5,6,200,2,0.80,20,237.828028,391.079634,16875


## 5. Bàn giao

Mở `tuning_compare.ipynb` sau khi chạy đủ ba cấu hình. Notebook so sánh chỉ đọc JSON và dừng nếu protocol, danh sách ứng viên hoặc timestamp OOF khác nhau. Tham số tối ưu có thể khác giữa cấu hình và mô hình; không dùng một bộ tham số chung sau khi tuning.